# Telco Customer Churn Analysis

This notebook investigates customer churn through reproducible cleaning and analysis. All findings are associations and should not be interpreted as causal effects.

## Clean

**Goal.** Load the customer data, detect missing or incorrectly typed values, make defensible cleaning decisions, and verify the result. The original row count is recorded before any changes, and a fixed seed is defined for reproducibility in later analysis.

In [1]:
import pandas as pd

RANDOM_SEED = 42
DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv.xls"

df = pd.read_csv(DATA_PATH)
original_rows = len(df)
print(f"Loaded {original_rows:,} rows and {df.shape[1]} columns.")

Loaded 7,043 rows and 21 columns.


### Initial inspection

Standard null checks do not detect strings containing only whitespace, so text columns are also checked after trimming whitespace. This inspection shows that `TotalCharges` was loaded as text and contains 11 blank values.

In [2]:
text_columns = df.select_dtypes(include="object")
blank_counts = text_columns.apply(lambda column: column.str.strip().eq("").sum())

display(df.head())
display(df.dtypes.rename("dtype"))
display(blank_counts[blank_counts.gt(0)].rename("blank_values"))

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
Name: dtype, dtype: object

TotalCharges    11
Name: blank_values, dtype: int64

### Convert numeric text and handle missing charges

A reusable function examines every text column and converts it only when all its nonblank values are numeric. The 11 missing `TotalCharges` values all belong to customers with `tenure == 0`, so they represent new customers who have not accumulated charges. An assertion checks that explanation before the values are filled with 0.

In [3]:
def convert_numeric_text_columns(data):
    cleaned = data.copy()
    for column in cleaned.select_dtypes(include="object"):
        stripped = cleaned[column].str.strip()
        nonblank = stripped.ne("")
        converted = pd.to_numeric(stripped, errors="coerce")
        if nonblank.any() and converted[nonblank].notna().all():
            cleaned[column] = converted
    return cleaned

df = convert_numeric_text_columns(df)
missing_total_charges = df["TotalCharges"].isna()
assert df.loc[missing_total_charges, "tenure"].eq(0).all()
print(f"Zero-tenure customers with missing TotalCharges: {missing_total_charges.sum()}")
df.loc[missing_total_charges, "TotalCharges"] = 0

Zero-tenure customers with missing TotalCharges: 11


### Remove the identifier and preserve service status

`customerID` is unique for every row and does not describe customer behaviour, so it is removed. `No internet service` and `No phone service` remain distinct categories because lacking a service differs from having the service and declining an add-on.

In [4]:
assert df["customerID"].nunique() == len(df)
no_internet_columns = df.columns[df.eq("No internet service").any()].tolist()
no_phone_columns = df.columns[df.eq("No phone service").any()].tolist()
df = df.drop(columns="customerID")

print("No internet service columns:", no_internet_columns)
print("No phone service columns:", no_phone_columns)

No internet service columns: ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
No phone service columns: ['MultipleLines']


### Clean-data validation

The final checks confirm that no rows were lost, no missing values remain, `TotalCharges` is numeric, and the target still uses clear `Yes`/`No` labels.

In [5]:
churn_rate = df["Churn"].eq("Yes").mean()
assert len(df) == original_rows
assert df.isna().sum().sum() == 0
assert pd.api.types.is_numeric_dtype(df["TotalCharges"])
assert set(df["Churn"].unique()) == {"Yes", "No"}

print(f"Rows: {original_rows:,} before, {len(df):,} after")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Churn rate: {churn_rate:.2%}")
display(df.dtypes.rename("dtype"))

Rows: 7,043 before, 7,043 after
Missing values: 0
Churn rate: 26.54%


gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
Name: dtype, dtype: object